## cài đặt thư viện


In [ ]:
# Nếu môi trường chưa có thư viện, bỏ comment dòng dưới để cài đặt
# %pip install -q pandas numpy scikit-learn matplotlib seaborn joblib

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import json

from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix, make_scorer
)
from sklearn.base import BaseEstimator, TransformerMixin, clone

RANDOM_STATE = 42
EXPERIMENT_NAME = "E09"
RESULT_DIR = Path(r"C:\homework\Machine Learning\SGU26K2_MayHoc_DCT124C3_Nhom4_RT-IoT2022\final_IoT\sample_project\result\exps\exp3\E09_RF")
RESULT_DIR.mkdir(parents=True, exist_ok=True)
TRAIN_PATH = Path(r"C:\homework\Machine Learning\SGU26K2_MayHoc_DCT124C3_Nhom4_RT-IoT2022\final_IoT\sample_project\result\exps\exp3\E09_RF\E09_Preprocessing\train.csv.gz")
VALID_PATH = Path(r"C:\homework\Machine Learning\SGU26K2_MayHoc_DCT124C3_Nhom4_RT-IoT2022\final_IoT\sample_project\result\exps\exp3\E09_RF\E09_Preprocessing\validation.csv.gz")
TARGET = "Attack_type"

def find_data_file(filename: str) -> Path:
    # TÃ¬m file theo vÃ²ng lÃªn thÆ° mÃ»c cha, khÃ´ng phÃ¹ thuá»™c notebook Ä‘Æ°á»£c cháº¡y tá»« Ä‘Ã¢u.
    cwd = Path.cwd().resolve()
    candidates = [
        cwd / filename,
        cwd / "splitted data" / filename,
        cwd / ".." / "splitted data" / filename,
        cwd / ".." / ".." / "splitted data" / filename,
    ]
    # Search project data folders from the notebook directory upwards.
    for parent in (cwd, *cwd.parents):
        candidates.extend([
            parent / "splitted data" / filename,
            parent / "preprocessing" / "data" / filename,
        ])

    for path in candidates:
        path = path.resolve()
        if path.is_file():
            return path

    searched = "\n".join(f"  - {path.resolve()}" for path in candidates)
    raise FileNotFoundError(f"KhÃ´ng tÃ¬m tháº¥y {filename}. ÄÃ£ kiá»ƒm tra:\n{searched}")

if not TRAIN_PATH.is_file() or not VALID_PATH.is_file():
    raise FileNotFoundError("Hãy chạy toàn bộ preprocessing/E09_RF.ipynb trước khi huấn luyện E09.")


## đọc dữ liệu và kiểm tra cấu trúc


In [ ]:
train_df = pd.read_csv(TRAIN_PATH)
valid_df = pd.read_csv(VALID_PATH)

print("Train shape:", train_df.shape)
print("Validation shape:", valid_df.shape)
print("\\nCác cột train:")
print(train_df.columns.tolist())

if TARGET not in train_df.columns or TARGET not in valid_df.columns:
    raise ValueError(f"Không tìm thấy cột nhãn '{TARGET}' trong cả hai tập dữ liệu.")

# Chỉ tách X/y, không loại bỏ hay biến đổi đặc trưng.
X_train = train_df.drop(columns=[TARGET]).copy()
y_train = train_df[TARGET].copy()
X_valid = valid_df.drop(columns=[TARGET]).copy()
y_valid = valid_df[TARGET].copy()

missing_cols = [c for c in X_train.columns if c not in X_valid.columns]
extra_cols = [c for c in X_valid.columns if c not in X_train.columns]
if missing_cols or extra_cols:
    raise ValueError(
        f"Cột không khớp giữa train và validation. "
        f"Thiếu ở validation: {missing_cols}; dư ở validation: {extra_cols}"
    )
X_valid = X_valid[X_train.columns]

# Không tự động điền giá trị thiếu vì yêu cầu không xử lý dữ liệu.
if X_train.isna().any().any() or X_valid.isna().any().any():
    raise ValueError("Dữ liệu có giá trị thiếu. Notebook không tự impute; cần xử lý nguồn dữ liệu trước.")

print("\\nSố lớp train:", y_train.nunique())
print("Phân bố lớp train:")
display(y_train.value_counts())

## Kiểm tra phụ thuộc thông tin cổng và loại bỏ cột cổng


In [ ]:
from scipy.stats import chi2_contingency
from sklearn.feature_selection import mutual_info_classif

# Kiểm tra mức phụ thuộc của hai cột cổng trước khi loại bỏ.
PORT_COLS = ["id.orig_p", "id.resp_p"]
present_port_cols = [col for col in PORT_COLS if col in X_train.columns]
port_audit = []

for col in present_port_cols:
    counts = X_train[col].value_counts(dropna=False)
    contingency = pd.crosstab(X_train[col], y_train)
    chi2, chi2_pvalue, _, _ = chi2_contingency(contingency)

    feature_values = X_train[[col]].copy()
    feature_values[col] = pd.factorize(feature_values[col])[0]

    mutual_info = mutual_info_classif(
        feature_values,
        y_train,
        discrete_features=True,
        random_state=RANDOM_STATE,
    )[0]

    n_classes = y_train.nunique()
    normalized_mutual_info = mutual_info / np.log(n_classes) if n_classes > 1 else 0.0

    port_audit.append({
        "feature": col,
        "n_unique": X_train[col].nunique(dropna=False),
        "top_value_share": counts.iloc[0] / len(X_train),
        "mutual_information": mutual_info,
        "normalized_mutual_information": normalized_mutual_info,
        "chi2_pvalue": chi2_pvalue,
    })

if port_audit:
    display(pd.DataFrame(port_audit))

    X_train = X_train.drop(columns=present_port_cols)
    X_valid = X_valid.drop(columns=present_port_cols)

    print("Đã loại bỏ:", present_port_cols)
else:
    print("Không còn cột cổng trong dữ liệu:", PORT_COLS)

print("X_train sau xử lý:", X_train.shape)
print("X_valid sau xử lý:", X_valid.shape)

## Xây dựng pipelin cho E09


In [ ]:
numeric_cols = X_train.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_cols = [c for c in X_train.columns if c not in numeric_cols]

preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
    ],
    remainder="passthrough"
)

rf_e09 = RandomForestClassifier(
    n_estimators=100,
    random_state=RANDOM_STATE,
    class_weight="balanced_subsample",
    n_jobs=-1
)

e09_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", rf_e09)
])

print("Số cột số:", len(numeric_cols))
print("Số cột phân loại:", len(categorical_cols))
print("Các cột phân loại:", categorical_cols)
print(e09_model)

## Cross-validation cho 3-fold trên tập train


In [ ]:
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

scoring = {
    "accuracy": "accuracy",
    "macro_f1": make_scorer(f1_score, average="macro", zero_division=0),
    "weighted_f1": make_scorer(f1_score, average="weighted", zero_division=0),
    "macro_precision": make_scorer(precision_score, average="macro", zero_division=0),
    "macro_recall": make_scorer(recall_score, average="macro", zero_division=0),
}

cv_results = cross_validate(
    e09_model,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    return_train_score=False,
    n_jobs=1,
    error_score="raise"
)

cv_summary = pd.DataFrame({
    "metric": list(scoring.keys()),
    "mean": [cv_results[f"test_{m}"].mean() for m in scoring],
    "std": [cv_results[f"test_{m}"].std(ddof=1) for m in scoring],
    "fold_1": [cv_results[f"test_{m}"][0] for m in scoring],
    "fold_2": [cv_results[f"test_{m}"][1] for m in scoring],
    "fold_3": [cv_results[f"test_{m}"][2] for m in scoring],
})
display(cv_summary.style.format({c: "{:.4f}" for c in cv_summary.columns if c != "metric"}))

## Huấn luyện lại toàn bộ tập train và đánh giá Validation


In [ ]:
# Fit lại mô hình trên toàn bộ tập train
e09_model.fit(X_train, y_train)

# Dự đoán validation
y_pred = e09_model.predict(X_valid)

validation_scores = {
    "accuracy": accuracy_score(y_valid, y_pred),
    "macro_f1": f1_score(y_valid, y_pred, average="macro", zero_division=0),
    "weighted_f1": f1_score(y_valid, y_pred, average="weighted", zero_division=0),
    "macro_precision": precision_score(y_valid, y_pred, average="macro", zero_division=0),
    "macro_recall": recall_score(y_valid, y_pred, average="macro", zero_division=0),
}

validation_summary = pd.DataFrame(
    [{"model": "E09_RF_processed", **validation_scores}]
)
display(validation_summary.style.format({
    k: "{:.4f}" for k in validation_scores
}))

print(classification_report(y_valid, y_pred, zero_division=0))

## Confusion matrix


In [ ]:
labels = e09_model.named_steps["classifier"].classes_
cm = confusion_matrix(y_valid, y_pred, labels=labels)

plt.figure(figsize=(max(9, len(labels) * 0.7), max(7, len(labels) * 0.6)))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=labels, yticklabels=labels
)
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.title("E09 – Confusion Matrix (Validation)")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(RESULT_DIR / "E09_validation_confusion_matrix.png", dpi=180, bbox_inches="tight")
plt.show()

### Save metric outputs


In [ ]:
labels = list(e08_model.named_steps["classifier"].classes_) if EXPERIMENT_NAME == "E08" else list(e09_model.named_steps["classifier"].classes_)
model_name = "E08_RF" if EXPERIMENT_NAME == "E08" else "E09_RF"
fold_metrics = pd.DataFrame({"fold": np.arange(1, len(cv_results["test_accuracy"]) + 1), **{metric: cv_results[f"test_{metric}"] for metric in scoring}})
fold_metrics.to_csv(RESULT_DIR / f"{EXPERIMENT_NAME}_cv_folds.csv", index=False, encoding="utf-8-sig")
cv_summary.to_csv(RESULT_DIR / f"{EXPERIMENT_NAME}_cv_summary.csv", index=False, encoding="utf-8-sig")
validation_summary.to_csv(RESULT_DIR / f"{EXPERIMENT_NAME}_validation_metrics.csv", index=False, encoding="utf-8-sig")
report = classification_report(y_valid, y_pred, output_dict=True, zero_division=0)
per_class = pd.DataFrame([{"class": label, "precision": report[str(label)]["precision"], "recall": report[str(label)]["recall"], "f1_score": report[str(label)]["f1-score"], "support": int(report[str(label)]["support"])} for label in labels])
per_class.to_csv(RESULT_DIR / f"{EXPERIMENT_NAME}_validation_per_class.csv", index=False, encoding="utf-8-sig")
pd.DataFrame(cm, index=pd.Index(labels, name="Actual"), columns=labels).to_csv(RESULT_DIR / f"{EXPERIMENT_NAME}_validation_confusion_matrix.csv", encoding="utf-8-sig")
pd.DataFrame({"y_true": y_valid.to_numpy(), "y_pred": y_pred}).to_csv(RESULT_DIR / f"{EXPERIMENT_NAME}_validation_predictions.csv", index=False, encoding="utf-8-sig")
with (RESULT_DIR / f"{EXPERIMENT_NAME}_config.json").open("w", encoding="utf-8") as f:
    json.dump({"experiment": EXPERIMENT_NAME, "model": model_name, "n_estimators": 100, "class_weight": "balanced_subsample", "random_state": RANDOM_STATE, "train_path": str(TRAIN_PATH), "validation_path": str(VALID_PATH)}, f, ensure_ascii=False, indent=2)
print("Saved CV, Validation and per-class evidence to:", RESULT_DIR)
